# 02 — Paired Bootstrap Test for Per-Instance Metric Scores

Use a paired bootstrap test when each test item has a numeric score for Model A and Model B.

Examples include document-level ROUGE, example-level F1, semantic similarity scores, or other per-instance metrics.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make src/ importable when running from the notebooks folder.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT / 'src'))

from significance_utils import *

DATA_DIR = PROJECT_ROOT / 'data' / 'dummy'
ALPHA = 0.05

## Student input

In [ ]:
ALPHA = 0.05
N_BOOTSTRAP = 10000
DATA_FILE = DATA_DIR / 'per_instance_scores.csv'

In [ ]:
df = pd.read_csv(DATA_FILE)
df.head()

In [ ]:
scores_a = df['model_a_rouge_like']
scores_b = df['model_b_rouge_like']

print(f'Model A mean score: {scores_a.mean():.4f}')
print(f'Model B mean score: {scores_b.mean():.4f}')
print(f'Difference B - A: {(scores_b - scores_a).mean():.4f}')

## Run a paired bootstrap test

The paired bootstrap resamples test items with replacement. Each resampled item keeps the Model A and Model B scores together.

In [ ]:
result = paired_bootstrap_test(
    scores_a=scores_a,
    scores_b=scores_b,
    n_bootstrap=N_BOOTSTRAP,
    alpha=ALPHA,
    seed=13,
)

pd.Series(result)

In [ ]:
print(interpret_p_value(result['p_value'], alpha=ALPHA))
print(f"{int((1-ALPHA)*100)}% bootstrap CI for difference: [{result['ci_low']:.4f}, {result['ci_high']:.4f}]")

## Visualize per-instance differences

In [ ]:
diffs = scores_b - scores_a
ax = diffs.plot(kind='hist', bins=20)
ax.set_xlabel('Per-instance difference: Model B - Model A')
ax.set_title('Distribution of paired score differences')
plt.show()

## Exercise questions

1. Why do we resample paired rows rather than sampling Model A scores and Model B scores independently?
2. What does it mean if the confidence interval includes zero?
3. Try reducing the number of rows in the dataframe. How does the result change?